#Ingest sprints.json file
1. Read the file using spark dataframe reader API
2. Define and enforce schema(preserve the nested structure)
3. Add Metadata columns:
    - Source File
    - Ingestion Timestamp
4. Write to bronze delta table


In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

#### Step 1 -  Read the CSV file using the dataframe reader API

In [0]:
#define schema
from pyspark.sql.types import *

sprints_schema = StructType ([ 
     StructField("date", DateType()),
     StructField("raceName", StringType()),
     StructField("round", IntegerType()),
     StructField("season", IntegerType()),
     StructField("url", StringType()),
     StructField("constructorId", StringType()),
     StructField("driverId", StringType()),
     StructField("grid", IntegerType()),
     StructField("laps", IntegerType()),
     StructField("number", IntegerType()),
     StructField("points", FloatType()),
     StructField("position", IntegerType()),
     StructField("positionText", StringType()),
     StructField("status", StringType())
 ])



In [0]:
sprints_df= (
    spark.read
    .format('json')
    .schema(sprints_schema)
    .option('mode','FAILFAST')
    .option('multiLine','true')
    .load(source_file)
    )

#### Step 2 - Add Metadata columns:
  - Source File
  - Ingestion Timestamp

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

#### Step 3 -  Write to bronze delta table

In [0]:
write_to_bronze(sprints_final_df, table_name, v_batch_id)